In [20]:
from build_everything import build_image_encoder, build_maskgit_v0, build_prompt_encoder, build_prompt_encoder, build_vqvae_single
from models.maskseg import MaskSeg
from models.maskgit import MaskGIT
from datasets.coco_lvis import LvisDataset
from trainer import MaskLevelDataset
from utils.clicker import Clicker

from typing import Optional
import torch
import torch.nn.functional as F
from torch.utils.data.dataloader import DataLoader

In [2]:
def build_maskseg(vqvae_checkpoint_path: Optional[str] = None, maskgit_checkpoint_path: Optional[str] = None, sam_checkpoint_path: Optional[str] = None) -> MaskSeg:
    vqvae = build_vqvae_single(vqvae_checkpoint_path)

    prompt_encoder = build_prompt_encoder(sam_checkpoint_path)
    image_encoder = build_image_encoder(sam_checkpoint_path)

    maskgit = build_maskgit_v0(vqvae, maskgit_checkpoint_path)

    maskseg = MaskSeg(maskgit=maskgit, 
                      prompt_encoder=prompt_encoder, 
                      image_encoder=image_encoder, 
                      freeze_prompt_encoder=True)

    return maskseg

In [3]:
maskseg = build_maskseg()

In [5]:
maskseg.load_state_dict(torch.load('ckpt/maskseg_old_epoch_1.pth'))

<All keys matched successfully>

In [18]:
maskseg.to('cuda:0')

MaskSeg(
  (maskgit): MaskGIT(
    (vqvae): VQVAE_Single(
      (encoder): Encoder(
        (conv_in): Conv2d(1, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (down): ModuleList(
          (0-1): 2 x Module(
            (block): ModuleList(
              (0-1): 2 x ResnetBlock(
                (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
                (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
                (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
                (dropout): Identity()
                (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
                (nin_shortcut): Identity()
              )
            )
            (attn): ModuleList()
            (downsample): Downsample2x(
              (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))
            )
          )
          (2): Module(
            (block): ModuleList(
              (0): ResnetBlock(
         

In [10]:
dataset = LvisDataset(
    dataset_path='data/coco_lvis',
    split='train',
    img_split='train',
    stuff_prob=0.0,
)

In [13]:
mask_level_dataset = MaskLevelDataset(dataset, maskseg.image_encoder.sam_encoder, 'cuda:0')

In [16]:
def forward_pass(image_embed_sam, gt_mask_normalized, gt_mask, maskseg):
    """
    second stage non-interactive training

    image_embed_sam: (B, C_enc, H, W)
    gt_mask_normalized: (B, 1, H, W) -1 ~ 1 mask
    """
    gen_tokens = [1, 4, 16, 64, 256, 1024]


    image_embed = maskseg.image_encoder.adapt_conv(image_embed_sam).permute(0, 2, 3, 1) # (B, H/2, W/2, C)

    B, _, H, W = gt_mask_normalized.shape
    L = 1024
    C = 256
    N_iter = len(gen_tokens) # num of iterations for generating

    clickers = [Clicker(num_random_clicks=2) for _ in range(B)]
    for i, clicker in enumerate(clickers):
        clicker.set_gt_mask(gt_mask[i, 0].cpu().numpy())
        clicks = clicker.init_clicks()

    gt_idx = maskseg.maskgit.vqvae.img_to_idxBl(gt_mask_normalized.float())[-1] # (B, L)

    blank_tokens = torch.zeros_like(gt_idx) + maskseg.maskgit.vocab_size # (B, L)
    
    # shuffle positions
    positions = torch.randperm(L) # (L,)

    masks = torch.zeros_like(gt_idx) # (B, L)
    masks = masks.view(B, 1, L).repeat(1, N_iter, 1) # (B, N_iter, L)

    for i, num_tokens in enumerate(gen_tokens):
        masks[:, i, positions[:num_tokens]] = 1

    x = torch.where(
        masks == 1, 
        gt_idx.unsqueeze(1).expand(-1, N_iter, -1), 
        blank_tokens.unsqueeze(1).expand(-1, N_iter, -1)
    ) # (B, N_iter, L)
    x_pos = torch.arange(L).to('cuda:0').unsqueeze(0).unsqueeze(0).repeat(B, N_iter, 1) # (B, N_iter, L)

    dense_pe = maskseg.prompt_encoder.get_dense_pe() # (1, C_enc, H, W)
    dense_pe = dense_pe.permute(0, 2, 3, 1) # (1, H, W, C_enc)
    # dense_pe_transformed = self.maskseg.prompt_enc_adapter(dense_pe) # (1, H, W, C)

    image_embed = image_embed + dense_pe

    click_sam_format = [clicker.to_sam_format(pad_size=2) for clicker in clickers]
    click_pos_sam_format = torch.stack([click_tuple[0] for click_tuple in click_sam_format], dim=0) # (B * num_iter_clicks, 2)
    click_label_sam_format = torch.stack([click_tuple[1] for click_tuple in click_sam_format], dim=0) # (B * num_iter_clicks,)

    prompt_embed, _ = maskseg.prompt_encoder(
        points=(click_pos_sam_format.to('cuda:0'), click_label_sam_format.to('cuda:0')),
        boxes=None,
        masks=None
    ) # (B, L_click, C)

    prompt_embed = prompt_embed.view(B, 1, -1, C).repeat(1, N_iter, 1, 1) # (B, N_iter, L_click, C)
    prompt_embed = prompt_embed.view(B * N_iter, -1, C) # (B*N_iter, L_click, C)

    image_embed = image_embed.view(B, -1, C)
    image_embed = image_embed.unsqueeze(1).expand(-1, N_iter, -1, -1) # (B, N_iter, L, C)
    image_embed = image_embed.reshape(B * N_iter, -1, C) # (B*N_iter, L, C)

    logits = maskseg.maskgit(x.view(-1, L), x_pos.view(-1, L), image_embed, prompt_embed, dense_pe.view(1, L, C)) # (B*N_iter, L, vocal_size)
    
    gt_idx_for_loss = gt_idx.unsqueeze(1).expand(-1, N_iter, -1).reshape(B*N_iter*L) # (B*N_iter*L,)
    logits_for_loss = logits.reshape(B*N_iter*L, -1) # (B*N_iter*L, vocal_size)

    loss = F.cross_entropy(logits_for_loss, gt_idx_for_loss)

    return loss, logits

In [23]:
dataloader = DataLoader(mask_level_dataset, batch_size=1)

In [24]:
for image, image_embed_sam, single_mask_normalized, single_mask in dataloader:
    loss, logits = forward_pass(image_embed_sam=image_embed_sam, gt_mask_normalized=single_mask_normalized, gt_mask=single_mask, maskseg=maskseg)
    break

In [36]:
probs = F.softmax(logits / 0.5, dim=-1)
probs.shape

torch.Size([6, 1024, 4096])

In [38]:
samples = torch.multinomial(probs.view(-1, 4096), num_samples=1)
samples.shape

torch.Size([6144, 1])

In [41]:
samples = samples.reshape(6, 1024)

In [43]:
vqvae = build_vqvae_single('ckpt/vqvae_single.pth')
vqvae.to('cuda:0')

VQVAE_Single(
  (encoder): Encoder(
    (conv_in): Conv2d(1, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down): ModuleList(
      (0-1): 2 x Module(
        (block): ModuleList(
          (0-1): 2 x ResnetBlock(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
            (dropout): Identity()
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nin_shortcut): Identity()
          )
        )
        (attn): ModuleList()
        (downsample): Downsample2x(
          (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))
        )
      )
      (2): Module(
        (block): ModuleList(
          (0): ResnetBlock(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [48]:
imgs = vqvae.idxBl_to_img([samples], same_shape=True, last_one=True)

IndexError: list index out of range